# 06 — 3D Phylogenomic Hypercube: Gene A Divergence across 100 Species

**Goal:** Visualize sequence divergence of a multi-copy gene (Gene A, modeled after MROH6) across 100 species using an interactive 3D scatter plot.

**Connection to the pipeline:**
- Steps 01-02 showed that MROH6 has 596 copies in zebra finch with JC-corrected divergence of 0.53 (17.7x baseline)
- Step 03 attempted to test for positive selection via dN/dS
- Step 05 modeled RNA vs DNA duplication dynamics
- **This step** asks: what would this pattern look like across 100 species, arranged by phylogenetic distance?

**Model:**
- X-axis: 100 species sorted by phylogenetic distance from a reference
- Y-axis: Locus index (0 = ancestral/ancient copy, 1+ = paralogs ranked by divergence)
- Z-axis: Sequence divergence score (Ks/dS-like metric)
- Color: Divergence (Viridis scale)
- Symbol: Ancient vs Paralog
- Size: Confidence (inversely related to divergence)

**Inputs:** Synthetic data parameterized from Steps 2 & 5 (mutation rate empirics)
**Outputs:** Interactive 3D Plotly figure, DataFrame for Dash dashboard

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_PROC = Path('../data/processed')
RESULTS   = Path('../results')

# Ensure output directories exist
(RESULTS / 'tables').mkdir(parents=True, exist_ok=True)
(RESULTS / 'figures').mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


## 6a. Define model parameters

We parameterize the synthetic data using empirical findings from earlier steps:
- **Divergence range 0.1–0.8:** Our MROH6 JC-corrected mean was 0.53 with median 0.39, so 0.1–0.8 captures the observed distribution
- **1–5 paralogs per species:** MROH6 shows variable copy number across chromosomes; 1–5 is conservative for a cross-species model
- **Phylogenetic distance sorting:** Species ordered by increasing evolutionary distance from the reference

In [2]:
# ── Model parameters ────────────────────────────────────────────────────────
N_SPECIES        = 100                   # Number of independent species
MIN_PARALOGS     = 1                     # Minimum paralogs per species
MAX_PARALOGS     = 5                     # Maximum paralogs per species
DIV_LOW          = 0.1                   # Minimum paralog divergence (Ks-like)
DIV_HIGH         = 0.8                   # Maximum paralog divergence
ANCESTRAL_DIV    = 0.0                   # Ancient locus divergence (reference)
SEED             = 42                    # Reproducibility

# Connection to empirical data
MROH6_JC_MEAN    = 0.5295                # From notebook 02
MROH6_BASELINE   = 0.03                  # Genomic baseline
FOLD_ELEVATION   = MROH6_JC_MEAN / MROH6_BASELINE  # ~17.7x

print(f'Species: {N_SPECIES}')
print(f'Paralogs per species: {MIN_PARALOGS}-{MAX_PARALOGS}')
print(f'Divergence range: [{DIV_LOW}, {DIV_HIGH}]')
print(f'Empirical fold elevation: {FOLD_ELEVATION:.1f}x (from MROH6 analysis)')

Species: 100
Paralogs per species: 1-5
Divergence range: [0.1, 0.8]
Empirical fold elevation: 17.6x (from MROH6 analysis)


## 6b. Generate synthetic phylogenomic data

For each of the 100 species:
1. Create one **ancient locus** (divergence = 0, confidence = 1.0)
2. Create 1–5 random **paralogs** with divergences drawn from a distribution shaped by phylogenetic distance
3. Rank paralogs by divergence (locus index 1, 2, 3, ...)
4. Compute confidence as `1.0 - divergence * 0.5` (higher divergence → lower alignment confidence)

In [3]:
# ── Synthetic data generation ────────────────────────────────────────────────
np.random.seed(SEED)

rows = []

for sp_idx in range(N_SPECIES):
    species_name = f'Species_{sp_idx:03d}'
    
    # ── Ancient locus (reference ortholog) ──
    rows.append({
        'Species':      species_name,
        'Species_Index': sp_idx,
        'Locus_Type':   'Ancient',
        'Locus_Index':  0,
        'Divergence':   ANCESTRAL_DIV,
        'Confidence':   1.0,
    })
    
    # ── Paralogs: 1-5 per species ──
    n_paralogs = np.random.randint(MIN_PARALOGS, MAX_PARALOGS + 1)
    
    # Divergence increases with phylogenetic distance (species index)
    # Species closer to index 0 tend to have lower divergence paralogs
    phylo_factor = (sp_idx + 1) / N_SPECIES  # 0.01 to 1.0
    
    # Draw divergences: beta distribution shaped by phylogenetic distance
    # Closer species → more paralogs cluster at low divergence
    # Distant species → paralogs spread across full range
    alpha_param = 2.0
    beta_param  = max(1.0, 5.0 - 4.0 * phylo_factor)  # decreases with distance
    raw_divs = np.random.beta(alpha_param, beta_param, size=n_paralogs)
    paralog_divs = DIV_LOW + raw_divs * (DIV_HIGH - DIV_LOW)
    
    # Sort paralogs by divergence (ranked)
    paralog_divs = np.sort(paralog_divs)
    
    for p_idx, div in enumerate(paralog_divs, start=1):
        confidence = 1.0 - div * 0.5
        rows.append({
            'Species':      species_name,
            'Species_Index': sp_idx,
            'Locus_Type':   'Paralogues',
            'Locus_Index':  p_idx,
            'Divergence':   round(float(div), 4),
            'Confidence':   round(float(confidence), 4),
        })

# Build tidy DataFrame
df = pd.DataFrame(rows)

print(f'Generated DataFrame: {len(df)} rows x {len(df.columns)} columns')
print(f'  Ancient loci: {(df.Locus_Type == "Ancient").sum()}')
print(f'  Paralogs:     {(df.Locus_Type == "Paralogues").sum()}')
print(f'\nDivergence stats (paralogs only):')
print(df[df.Locus_Type == 'Paralogues']['Divergence'].describe())
print(f'\nFirst 10 rows:')
df.head(10)

Generated DataFrame: 379 rows x 6 columns
  Ancient loci: 100
  Paralogs:     279

Divergence stats (paralogs only):
count    279.000000
mean       0.385243
std        0.160208
min        0.105700
25%        0.257200
50%        0.377300
75%        0.492000
max        0.762000
Name: Divergence, dtype: float64

First 10 rows:


,Species,Species_Index,Locus_Type,Locus_Index,Divergence,Confidence
0,Species_000,0,Ancient,0,0.0000,1.0000
1,Species_000,0,Paralogues,1,0.1057,0.9471
2,Species_000,0,Paralogues,2,0.1710,0.9145
3,Species_000,0,Paralogues,3,0.2554,0.8723
4,Species_000,0,Paralogues,4,0.2560,0.8720
5,Species_001,1,Ancient,0,0.0000,1.0000
6,Species_001,1,Paralogues,1,0.2629,0.8685
7,Species_001,1,Paralogues,2,0.2702,0.8649
8,Species_002,2,Ancient,0,0.0000,1.0000
9,Species_002,2,Paralogues,1,0.2419,0.8790


## 6c. Exploratory analysis of the synthetic data

Before building the 3D visualization, let's understand the data structure.

In [4]:
# ── Summary statistics per species ──────────────────────────────────────────
species_summary = df.groupby('Species').agg(
    n_loci       = ('Locus_Index', 'count'),
    n_paralogs   = ('Locus_Type', lambda x: (x == 'Paralogues').sum()),
    mean_div     = ('Divergence', 'mean'),
    max_div      = ('Divergence', 'max'),
    mean_conf    = ('Confidence', 'mean'),
).reset_index()

print('Species summary statistics:')
print(species_summary.describe())
print(f'\nTotal loci across all species: {len(df)}')
print(f'Mean paralogs per species: {species_summary.n_paralogs.mean():.1f}')
print(f'Mean divergence across species: {species_summary.mean_div.mean():.4f}')

Species summary statistics:
           n_loci  n_paralogs    mean_div     max_div   mean_conf
count  100.000000  100.000000  100.000000  100.000000  100.000000
mean     3.790000    2.790000    0.263783    0.470755    0.868107
std      1.401983    1.401983    0.099751    0.164268    0.049874
min      2.000000    1.000000    0.057900    0.115800    0.757267
25%      2.750000    1.750000    0.189963    0.339750    0.831769
50%      4.000000    3.000000    0.255550    0.481800    0.872225
75%      5.000000    4.000000    0.336419    0.594675    0.905012
max      6.000000    5.000000    0.485450    0.762000    0.971050

Total loci across all species: 379
Mean paralogs per species: 2.8
Mean divergence across species: 0.2638


In [5]:
# ── 2D distribution plots ──────────────────────────────────────────────────
import plotly.subplots as sp

fig_2d = sp.make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'A. Divergence distribution (paralogs)',
        'B. Paralogs per species',
        'C. Divergence vs Species Index',
        'D. Confidence vs Divergence'
    )
)

# A: Divergence histogram
paralog_divs = df[df.Locus_Type == 'Paralogues']['Divergence']
fig_2d.add_trace(
    go.Histogram(x=paralog_divs, nbinsx=30, marker_color='#440154', name='Paralogs'),
    row=1, col=1
)

# B: Paralogs per species
fig_2d.add_trace(
    go.Histogram(x=species_summary['n_paralogs'], nbinsx=5, marker_color='#31688e', name='Count'),
    row=1, col=2
)

# C: Divergence vs species index (phylogenetic trend)
par_df = df[df.Locus_Type == 'Paralogues']
fig_2d.add_trace(
    go.Scatter(
        x=par_df['Species_Index'], y=par_df['Divergence'],
        mode='markers', marker=dict(size=4, color=par_df['Divergence'],
        colorscale='Viridis', opacity=0.6), name='Paralogs'
    ),
    row=2, col=1
)

# D: Confidence vs divergence
fig_2d.add_trace(
    go.Scatter(
        x=df['Divergence'], y=df['Confidence'],
        mode='markers', marker=dict(size=5, color=df['Divergence'],
        colorscale='Viridis', opacity=0.6), name='All loci'
    ),
    row=2, col=2
)

fig_2d.update_layout(
    height=700, width=900,
    title_text='Synthetic Phylogenomic Data: Exploratory Analysis',
    showlegend=False
)
fig_2d.update_xaxes(title_text='Divergence', row=1, col=1)
fig_2d.update_xaxes(title_text='Number of paralogs', row=1, col=2)
fig_2d.update_xaxes(title_text='Species Index (phylo distance)', row=2, col=1)
fig_2d.update_xaxes(title_text='Divergence', row=2, col=2)
fig_2d.update_yaxes(title_text='Count', row=1, col=1)
fig_2d.update_yaxes(title_text='Count', row=1, col=2)
fig_2d.update_yaxes(title_text='Divergence', row=2, col=1)
fig_2d.update_yaxes(title_text='Confidence', row=2, col=2)
fig_2d.show()

## 6d. 3D Phylogenomic Hypercube visualization

This is the core visualization — an interactive 3D scatter plot where:
- **X = Species** (sorted by phylogenetic distance)
- **Y = Locus index** (0 = ancient ortholog, 1+ = paralogs ranked by divergence)
- **Z = Sequence divergence** (Ks/dS-like score)
- **Color = Divergence** (Viridis: dark = conserved, yellow = diverged)
- **Symbol = Locus type** (diamond = Ancient, circle = Paralog)
- **Size = Confidence** (larger = more confident alignment)

In [6]:
# ── 3D Phylogenomic Matrix ──────────────────────────────────────────────────

fig = px.scatter_3d(
    df,
    x='Species',
    y='Locus_Index',
    z='Divergence',
    color='Divergence',
    symbol='Locus_Type',
    size='Confidence',
    color_continuous_scale='Viridis',
    title='3D Phylogenomic Matrix: Gene A Divergence across 100 Species',
    labels={
        'Species':     'Species',
        'Locus_Index': 'Locus (0=Ancient)',
        'Divergence':  'Sequence Distance',
    },
    hover_data=['Species', 'Locus_Type', 'Locus_Index', 'Divergence', 'Confidence'],
)

# Optimize layout for 100 species
fig.update_layout(
    scene=dict(
        xaxis=dict(
            title='Species',
            nticks=10,
            showticklabels=False,   # Too many species for labels
        ),
        yaxis=dict(title='Locus (0=Ancient)'),
        zaxis=dict(title='Sequence Distance'),
    ),
    width=1000,
    height=750,
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()

## 6e. Connection to MROH6 empirical findings

How does this synthetic model relate to the real MROH6 data?

| Metric | MROH6 (Zebra Finch) | Synthetic Model |
|--------|--------------------|-----------------|
| Copies analyzed | 596 | ~400 (100 species x ~4 avg) |
| Mean divergence | 0.53 (JC) | ~0.35 (designed range 0.1-0.8) |
| Baseline | 0.03 | 0.0 (ancient reference) |
| Fold elevation | 17.7x | Variable by species |
| Ts/Tv ratio | 0.89 | Not modeled (future extension) |

The synthetic model extends the single-species MROH6 analysis to a cross-species
phylogenomic perspective, showing how paralog divergence patterns would appear
across an evolutionary radiation.

In [7]:
# ── Overlay empirical MROH6 divergence distribution onto synthetic model ────

# Create comparison: what if one species is zebra finch with MROH6-like data?
print('Comparing synthetic model to MROH6 empirical data:')
print(f'  MROH6 JC-corrected mean divergence: {MROH6_JC_MEAN:.4f}')
print(f'  MROH6 genomic baseline:             {MROH6_BASELINE}')
print(f'  Fold elevation:                     {FOLD_ELEVATION:.1f}x')
print()

# Where does MROH6 fall in our synthetic divergence distribution?
synthetic_paralogs = df[df.Locus_Type == 'Paralogues']['Divergence']
pct_below_mroh6 = (synthetic_paralogs < MROH6_JC_MEAN).mean() * 100
print(f'  {pct_below_mroh6:.1f}% of synthetic paralogs have divergence < MROH6 mean ({MROH6_JC_MEAN:.2f})')
print(f'  This suggests MROH6 divergence is {'typical' if 30 < pct_below_mroh6 < 70 else 'extreme'} '
      f'relative to the synthetic model')

# Highlight: add MROH6-like reference plane to the 3D plot
fig_annotated = go.Figure(fig)

# Add a horizontal reference plane at MROH6 mean divergence
x_range = list(range(N_SPECIES))
y_range = list(range(MAX_PARALOGS + 1))
xx, yy = np.meshgrid(x_range[::10], y_range)  # subsample x for performance
zz = np.full_like(xx, MROH6_JC_MEAN, dtype=float)

fig_annotated.add_trace(go.Surface(
    x=xx, y=yy, z=zz,
    opacity=0.2, colorscale=[[0, 'red'], [1, 'red']],
    showscale=False, name=f'MROH6 mean (JC={MROH6_JC_MEAN:.2f})'
))

fig_annotated.update_layout(
    title='3D Phylogenomic Matrix with MROH6 Empirical Reference Plane',
    width=1000, height=750,
)
fig_annotated.show()

Comparing synthetic model to MROH6 empirical data:
  MROH6 JC-corrected mean divergence: 0.5295
  MROH6 genomic baseline:             0.03
  Fold elevation:                     17.6x

  81.7% of synthetic paralogs have divergence < MROH6 mean (0.53)
  This suggests MROH6 divergence is extreme relative to the synthetic model


## 6f. Phylogenetic trend analysis

Does divergence increase with phylogenetic distance? This is the key prediction:
- **DNA-only duplication:** Divergence should correlate linearly with phylogenetic distance
- **RNA-mediated duplication:** Divergence should be elevated across all distances (elevated intercept)
  and potentially show non-linear accumulation (RT error rate differs from DNA polymerase)

In [8]:
# ── Phylogenetic trend: mean divergence vs species index ────────────────────
from scipy import stats as sp_stats

par_summary = df[df.Locus_Type == 'Paralogues'].groupby('Species_Index').agg(
    mean_div  = ('Divergence', 'mean'),
    max_div   = ('Divergence', 'max'),
    n_paralogs= ('Divergence', 'count'),
).reset_index()

# Linear regression
slope, intercept, r_val, p_val, se = sp_stats.linregress(
    par_summary['Species_Index'], par_summary['mean_div']
)

print(f'Linear trend (divergence ~ phylo distance):')
print(f'  slope     = {slope:.6f} per species index unit')
print(f'  intercept = {intercept:.4f}')
print(f'  R-squared = {r_val**2:.4f}')
print(f'  p-value   = {p_val:.2e}')

# Plot
fig_trend = px.scatter(
    par_summary, x='Species_Index', y='mean_div',
    size='n_paralogs', color='max_div',
    color_continuous_scale='Viridis',
    labels={'Species_Index': 'Species Index (phylo distance)',
            'mean_div': 'Mean Paralog Divergence',
            'max_div': 'Max Divergence'},
    title=f'Divergence vs Phylogenetic Distance (R²={r_val**2:.3f}, p={p_val:.1e})'
)

# Add regression line
x_fit = np.array([0, N_SPECIES-1])
y_fit = slope * x_fit + intercept
fig_trend.add_trace(go.Scatter(
    x=x_fit, y=y_fit, mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name=f'Linear fit (slope={slope:.4f})'
))

# Add MROH6 reference line
fig_trend.add_hline(
    y=MROH6_JC_MEAN, line_dash='dot', line_color='orange',
    annotation_text=f'MROH6 JC mean ({MROH6_JC_MEAN:.2f})',
)

fig_trend.update_layout(height=500, width=800)
fig_trend.show()

Linear trend (divergence ~ phylo distance):
  slope     = 0.002199 per species index unit
  intercept = 0.2680
  R-squared = 0.2757
  p-value   = 2.04e-08


## 6g. Save outputs for Dash dashboard

In [9]:
# ── Save DataFrame for dashboard ────────────────────────────────────────────
output_csv = RESULTS / 'tables' / 'phylogenomic_hypercube_data.csv'
df.to_csv(output_csv, index=False)
print(f'Saved phylogenomic data: {output_csv}')

# Save species summary
species_summary.to_csv(RESULTS / 'tables' / 'phylogenomic_species_summary.csv', index=False)
print(f'Saved species summary')

# Save phylogenetic trend data
par_summary.to_csv(RESULTS / 'tables' / 'phylogenomic_trend_data.csv', index=False)
print(f'Saved trend data')

# Save the 3D figure as HTML for standalone viewing
html_path = RESULTS / 'figures' / 'phylogenomic_3d_hypercube.html'
fig.write_html(str(html_path))
print(f'Saved interactive HTML: {html_path}')

Saved phylogenomic data: ../results/tables/phylogenomic_hypercube_data.csv
Saved species summary
Saved trend data
Saved interactive HTML: ../results/figures/phylogenomic_3d_hypercube.html


In [10]:
# ── Summary ─────────────────────────────────────────────────────────────────
print('=' * 60)
print('PHYLOGENOMIC HYPERCUBE SUMMARY')
print('=' * 60)
print(f'Species modeled:           {N_SPECIES}')
print(f'Total data points:         {len(df)}')
print(f'  Ancient loci:            {(df.Locus_Type == "Ancient").sum()}')
print(f'  Paralogs:                {(df.Locus_Type == "Paralogues").sum()}')
print(f'Mean paralogs per species: {species_summary.n_paralogs.mean():.1f}')
print(f'Divergence range:          [{df.Divergence.min():.2f}, {df.Divergence.max():.2f}]')
print(f'Mean paralog divergence:   {synthetic_paralogs.mean():.4f}')
print(f'Phylo trend R-squared:     {r_val**2:.4f}')
print(f'Phylo trend p-value:       {p_val:.2e}')
print()
print('Outputs:')
print(f'  CSV data:    {output_csv}')
print(f'  3D HTML:     {html_path}')
print()
print('=> Data ready for Dash dashboard (app.py)')

PHYLOGENOMIC HYPERCUBE SUMMARY
Species modeled:           100
Total data points:         379
  Ancient loci:            100
  Paralogs:                279
Mean paralogs per species: 2.8
Divergence range:          [0.00, 0.76]
Mean paralog divergence:   0.3852
Phylo trend R-squared:     0.2757
Phylo trend p-value:       2.04e-08

Outputs:
  CSV data:    ../results/tables/phylogenomic_hypercube_data.csv
  3D HTML:     ../results/figures/phylogenomic_3d_hypercube.html

=> Data ready for Dash dashboard (app.py)
